In [ ]:
# Module 4 Lab 01 - Python Database Connectivity

This notebook demonstrates SQL Server connectivity using `pyodbc`, SQLAlchemy, and pandas.

It is designed to run in a local virtual environment or in Google Colab. If SQL Server is not available, the notebook falls back to the local CSV sample so the rest of the practice still runs.

In [ ]:
# Install notebook dependencies.
# Run this cell in Colab or any fresh notebook environment.
import sys
import subprocess

core_packages = ["pandas", "numpy", "sqlalchemy", "python-dotenv", "openpyxl", "requests"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *core_packages])

# pyodbc is required only for SQL Server connectivity.
# If it cannot install in Colab, the notebook still continues with CSV fallback data.
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyodbc"])
except Exception as exc:
    print("pyodbc install skipped or failed. SQL Server cells may be skipped:", exc)

In [ ]:
import os
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

try:
    import pyodbc
except Exception as exc:
    pyodbc = None
    print("pyodbc is unavailable; SQL Server connection tests will be skipped:", exc)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
SAMPLE_CSV = DATA_DIR / "m4_raw_financial_transactions_sample.csv"

# In Colab, upload this repo folder or upload the CSV manually if needed.
print("Working directory:", BASE_DIR)

def load_sample_transactions():
    if SAMPLE_CSV.exists():
        return pd.read_csv(SAMPLE_CSV)
    print("Local CSV not found; using built-in fallback sample rows.")
    return pd.DataFrame([
        {"SourceSystem": "CoreBanking", "TransactionReference": "M4-0001", "TransactionDateText": "2026-06-01", "InstitutionCode": "MCB", "CounterpartyName": "Maseru Commercial Bank", "CurrencyCode": "LSL", "AmountText": "15000.00", "TransactionType": "Deposit", "Channel": "Branch", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "Payments", "TransactionReference": "M4-0003", "TransactionDateText": "2026/06/02", "InstitutionCode": "CPS", "CounterpartyName": "Cape Payment Services", "CurrencyCode": "ZAR", "AmountText": "25000", "TransactionType": "Transfer", "Channel": "Clearing", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "Treasury", "TransactionReference": "M4-0004", "TransactionDateText": "2026-06-03", "InstitutionCode": "CBL", "CounterpartyName": "Central Bank of Lesotho", "CurrencyCode": "USD", "AmountText": "100000.00", "TransactionType": "FX Settlement", "Channel": "SWIFT", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "CoreBanking", "TransactionReference": "M4-0006", "TransactionDateText": None, "InstitutionCode": "MCB", "CounterpartyName": "Maseru Commercial Bank", "CurrencyCode": "LSL", "AmountText": "9100.00", "TransactionType": "Deposit", "Channel": "Branch", "LoadBatch": "BATCH-202606"},
        {"SourceSystem": "Treasury", "TransactionReference": "M4-0010", "TransactionDateText": "2026-06-08", "InstitutionCode": "CBL", "CounterpartyName": "Central Bank of Lesotho", "CurrencyCode": "usd", "AmountText": "bad_amount", "TransactionType": "FX Settlement", "Channel": "SWIFT", "LoadBatch": "BATCH-202606"},
    ])

In [ ]:
# Load local .env settings when available.
# In Colab, you can set these values manually in os.environ before running the connection cells.
load_dotenv(BASE_DIR / ".env")

DB_DRIVER = os.getenv("DB_DRIVER", "ODBC Driver 18 for SQL Server")
DB_SERVER = os.getenv("DB_SERVER", "localhost,1433")
DB_NAME = os.getenv("DB_NAME", "TrainingDB")
DB_USER = os.getenv("DB_USER", "sa")
DB_PASSWORD = os.getenv("DB_PASSWORD", "StrongPassw0rd!2026")
DB_TRUSTED = os.getenv("DB_TRUSTED", "no").lower() in ("yes", "true", "1")

def build_odbc_connection_string():
    parts = [
        f"DRIVER={{{DB_DRIVER}}}",
        f"SERVER={DB_SERVER}",
        f"DATABASE={DB_NAME}",
        "Encrypt=yes",
        "TrustServerCertificate=yes",
    ]
    if DB_TRUSTED:
        parts.append("Trusted_Connection=yes")
    else:
        parts.extend([f"UID={DB_USER}", f"PWD={DB_PASSWORD}"])
    return ";".join(parts) + ";"

def get_sqlalchemy_engine():
    quoted = quote_plus(build_odbc_connection_string())
    return create_engine(f"mssql+pyodbc:///?odbc_connect={quoted}", fast_executemany=True)

In [ ]:
# pyodbc connection test.
# If this fails in Colab, continue with the fallback data cells below.
if pyodbc is None:
    print("Skipping pyodbc test because pyodbc is not installed.")
else:
    try:
        conn = pyodbc.connect(build_odbc_connection_string(), timeout=5)
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM m4.RawFinancialTransactions;")
        print("Raw rows in SQL Server:", cursor.fetchone()[0])
        cursor.close()
        conn.close()
    except Exception as exc:
        print("SQL Server connection unavailable. Reason:", exc)

In [ ]:
# pandas + SQLAlchemy extraction.
# If SQL Server is unavailable, fall back to the local CSV sample.
query = """
SELECT
    RawTransactionID,
    SourceSystem,
    TransactionReference,
    TransactionDateText,
    InstitutionCode,
    CounterpartyName,
    CurrencyCode,
    AmountText,
    TransactionType,
    Channel,
    LoadBatch
FROM m4.RawFinancialTransactions
ORDER BY RawTransactionID;
"""

try:
    if pyodbc is None:
        raise RuntimeError("pyodbc is not installed")
    engine = get_sqlalchemy_engine()
    raw_df = pd.read_sql(query, engine)
    data_source = "SQL Server"
except Exception as exc:
    print("Falling back to CSV sample. Reason:", exc)
    raw_df = load_sample_transactions()
    raw_df.insert(0, "RawTransactionID", range(1, len(raw_df) + 1))
    data_source = "CSV fallback"

print("Data source:", data_source)
display(raw_df.head())
raw_df.info()